In [1]:
import sys
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from schemas.movie import Movie
from movie_ingestion.final_ingestion.vector_text import create_anchor_vector_text

# ============================================================
#  Edge-case test IDs for anchor text inspection
# ============================================================
#
#  79    — Hero: foreign film (original_title "Ying xiong" != title), full metadata
#  155897 — The Last Message: no plot_analysis (tests overview fallback, no elevator
#           pitch, no themes), has reception, non-standard rating "Not Rated"
#  242920 — The Glass House: non-standard rating "Not Rated" with full metadata
#  1386912 — The Rally: no LLM metadata at all + no maturity rating/reasoning
#            (tests overview fallback, empty maturity, no experience, no reception)

tmdb_ids = [
    79,         # Hero (foreign title, full metadata)
    155897,     # The Last Message (no plot_analysis, non-standard rating)
    242920,     # The Glass House (non-standard rating, full metadata)
    1386912,    # The Rally (no LLM metadata, no maturity)
]

# ============================================================

SEPARATOR = "=" * 80

for tmdb_id in tmdb_ids:
    try:
        movie = Movie.from_tmdb_id(tmdb_id)
    except LookupError as e:
        print(f"[{tmdb_id}] SKIPPED — {e}\n")
        continue

    title = movie.tmdb_data.title or "(no title)"

    print(SEPARATOR)
    print(f"  [{tmdb_id}] {title}")
    print(SEPARATOR)

    # --- Identity ---
    print(f"\n--- Identity ---")
    print(f"  title_with_original(): {movie.title_with_original()}")

    pa = movie.plot_analysis_metadata
    if pa:
        print(f"  elevator_pitch: {pa.elevator_pitch_with_justification.elevator_pitch}")
        print(f"  generalized_plot_overview: {pa.generalized_plot_overview}")
    else:
        print(f"  (no plot_analysis_metadata)")
        print(f"  imdb overview fallback: {movie.imdb_data.overview}")

    # --- Classification ---
    print(f"\n--- Classification ---")
    print(f"  deduplicated_genres(): {movie.deduplicated_genres()}")
    print(f"  imdb genres (raw): {movie.imdb_data.genres}")
    if pa:
        print(f"  genre_signatures (raw): {pa.genre_signatures}")
    print(f"  overall_keywords: {movie.imdb_data.overall_keywords}")

    if pa and pa.thematic_concepts:
        concepts = [t.concept_label for t in pa.thematic_concepts]
        print(f"  thematic_concepts: {concepts}")
    else:
        print(f"  thematic_concepts: (none)")

    soi = movie.source_of_inspiration_metadata
    if soi:
        print(f"  source_material: {soi.source_material}")
        print(f"  franchise_lineage: {soi.franchise_lineage}")
    else:
        print(f"  source_of_inspiration: (none)")

    # --- Experience ---
    print(f"\n--- Experience ---")
    ve = movie.viewer_experience_metadata
    if ve:
        print(f"  emotional_palette terms: {ve.emotional_palette.terms}")
    else:
        print(f"  emotional_palette: (none)")

    wc = movie.watch_context_metadata
    if wc:
        print(f"  key_movie_feature_draws terms: {wc.key_movie_feature_draws.terms}")
    else:
        print(f"  key_draws: (none)")

    # --- Context ---
    print(f"\n--- Context ---")
    print(f"  release_decade_bucket(): {movie.release_decade_bucket()}")
    print(f"  languages: {movie.imdb_data.languages}")
    print(f"  budget_bucket_for_era(): {movie.budget_bucket_for_era()}")

    # --- Maturity ---
    print(f"\n--- Maturity ---")
    print(f"  resolved_maturity_rating(): {movie.resolved_maturity_rating()}")
    print(f"  maturity_reasoning: {movie.imdb_data.maturity_reasoning}")
    print(f"  maturity_text_short(): {movie.maturity_text_short()}")

    # --- Reception ---
    print(f"\n--- Reception ---")
    rec = movie.reception_metadata
    if rec:
        print(f"  reception_summary: {rec.reception_summary}")
    else:
        print(f"  reception_summary: (none)")
    print(f"  reception_tier(): {movie.reception_tier()}")

    # --- Final anchor vector text ---
    print(f"\n{'- ' * 40}")
    print(f"ANCHOR VECTOR TEXT:")
    print(f"{'- ' * 40}")
    print(create_anchor_vector_text(movie))
    print()

  [79] Hero

--- Identity ---
  title_with_original(): Hero (Ying xiong)
  elevator_pitch: Assassin chooses unity over vengeance
  generalized_plot_overview: An orphaned assassin infiltrates a rival court to kill three legendary opponents and avenge a slaughter, but flashbacks reveal an earlier failed assassination where a warrior spared a unifying ruler; confronted with that ruler's vision, the assassin abandons vengeance, accepts sacrifice, and is executed—achieving the political unity the assassin once sought to destroy, underscoring sacrifice, the moral calculus of violence, and identity reshaped by duty.

--- Classification ---
  deduplicated_genres(): ['action', 'adventure', 'assassin drama', 'drama', 'historical action spectacle', 'wuxia martial epic']
  imdb genres (raw): ['Action', 'Adventure', 'Drama']
  genre_signatures (raw): ['wuxia martial epic', 'assassin drama', 'historical action spectacle']
  overall_keywords: ['Mandarin', 'Action Epic', 'Adventure Epic', 'Epic', 'Mar